# Bootstrap do ambiente — do zero ao ambiente pronto (idempotente)

Recria em **qualquer workspace** tudo que o pipeline e o job assumem que existe. Pode rodar quantas vezes quiser: cada passo é `IF NOT EXISTS` ou pula arquivo já presente.

| Passo | O que faz | Idempotência |
| --- | --- | --- |
| 1. Infraestrutura UC | catálogo + schemas `bronze/silver/gold/governanca/landing` + volumes `landing.files` e `landing.reference` + tabelas de governança | `CREATE ... IF NOT EXISTS` |
| 2. Dados de origem | parquets TLC via `url_template` dos contratos (motor genérico `motores.baixar_da_fonte`) | SHA-256 + footer Parquet; existente válido é pulado; landing antiga é copiada para as partições |
| 3. Referência TLC | `taxi_zone_lookup.csv` oficial para `landing/reference/` | sobrescreve com snapshot datado (converge para estado válido) |

Ordem de recriação do ambiente (fluxo do SDD): **este notebook → `databricks bundle deploy` → update do pipeline SDP → job `case_nyc_taxi_update_e_qualidade`** (qualidade + observabilidade). Widgets: `catalogo`, `ano`, `meses`. Partições: `categoria=<green|yellow>/mes_ano=<MM-AAAA>/`. A execução audita início/fim do lote, cada tentativa e resultado por partição, mesmo quando falha antes da observabilidade da pipeline.

In [ ]:
dbutils.widgets.text("catalogo", "case_nyc_taxi", "catalogo UC")
dbutils.widgets.text("ano", "2023", "ano")
dbutils.widgets.text("meses", "01,02,03,04,05", "meses (csv)")

catalogo = dbutils.widgets.get("catalogo")
ano = dbutils.widgets.get("ano")
meses = [m.strip().zfill(2) for m in dbutils.widgets.get("meses").split(",") if m.strip()]

import os
import sys

from pathlib import Path

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
candidatos = [Path(notebook_path), Path("/Workspace") / notebook_path.lstrip("/")]
REPO = None
for candidato in candidatos:
    for raiz in (candidato, *candidato.parents):
        if (raiz / "00_config" / "motores.py").is_file():
            REPO = str(raiz)
            break
    if REPO:
        break
if REPO is None:
    raise FileNotFoundError(f"Raiz do projeto nao localizada a partir do notebook: {notebook_path}")
for sub in ("00_config", "01_ingestao"):
    sys.path.insert(0, f"{REPO}/{sub}")

import motores
from atualizar_taxi_zone_lookup import atualizar as atualizar_lookup

from uuid import uuid4
from landing import normalizar_periodos, planejar_arquivos, listar_arquivos_landing

ano, meses = normalizar_periodos(ano, meses)
execucao_id = str(uuid4())
CONTRATOS = motores.listar_contratos()
print(f"Execução de ingestão: {execucao_id}")
print(f"Catalogo: {catalogo} | periodo: {ano}-{meses}")
print(f"Contratos de ingestao: {[c['dataset'] for c in CONTRATOS]}")

In [0]:
# 1) Infraestrutura UC — tudo IF NOT EXISTS: roda quantas vezes quiser
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")
for schema in ("bronze", "silver", "gold", "governanca"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema}")

# Volume de landing: no UC o caminho é catalog.schema.volume —
# /Volumes/case_nyc_taxi/landing/files = catálogo case_nyc_taxi, schema landing,
# volumes files (parquets) e reference (lookup TLC)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.landing")
for volume in ("files", "reference"):
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.landing.{volume}")

# Tabelas de governança do projeto (log_ingestao, dq_resultados, obs_metricas)
motores.criar_tabelas_governanca(catalogo, spark)

print("Infraestrutura OK:")
print(f"  catalogo: {catalogo}")
print("  schemas: bronze, silver, gold, governanca, landing")
print("  volumes: landing.files (parquets) e landing.reference (lookup TLC)")
print(f"  tabelas de governança em {catalogo}.governanca")

In [ ]:
# 2) Download ou migração da landing antiga, por categoria e mês/ano.
# Cada dataset registra o resumo antes de lançar FalhaIngestao. Processamos
# ambas as categorias para ter o resultado de TODAS as partições solicitadas.
import pandas as pd

resultados = []
for contrato in CONTRATOS:
    try:
        resultados.extend(motores.baixar_da_fonte(
            contrato, spark=spark, ano=ano, meses=meses, catalogo=catalogo,
            execucao_id=execucao_id,
        ))
    except motores.FalhaIngestao as exc:
        resultados.extend(exc.resultados)

display(pd.DataFrame(resultados))
if any(r["status"] == "erro" for r in resultados):
    raise motores.FalhaIngestao(resultados)

# 3) Lookup oficial de Taxi Zones (dimensão de referência da Gold).
info_lookup = atualizar_lookup(dbutils)
print(f"Lookup TLC: {info_lookup['rows']} zonas | sha256 {info_lookup['sha256'][:12]}… | {info_lookup['path']}")


In [ ]:
# Verificação das partições exatas pedidas, sem contar outros meses como sucesso.
faltantes = []
for contrato in CONTRATOS:
    esperados = {p["destino"] for p in planejar_arquivos(contrato, ano, meses)}
    publicados = set(listar_arquivos_landing(contrato))
    ausentes = sorted(esperados - publicados)
    faltantes.extend(ausentes)
    print(f"{contrato['dataset']}: {len(esperados) - len(ausentes)}/{len(esperados)} partições confirmadas")

if faltantes:
    raise RuntimeError(f"Bootstrap incompleto; partições ausentes: {faltantes}")
print(f"Bootstrap OK | execução {execucao_id}: infraestrutura + partições + lookup prontos.")
print("Próximo passo: databricks bundle run -t dev case_nyc_taxi_update_e_qualidade")
